# OAM-TCD Experiment Pipeline

This notebook runs the asset validation, training, tiled inference, locked evaluation, and baseline comparison workflow. It supports the Unified-Prompt SAM3 baseline (`A0` in constant and path names) and the granularity-aware configuration.

The committed notebook is intentionally inert. `FULL_LAUNCH_AUTHORIZED=False`, `LAUNCH=False`, `FORCE_INFERENCE=False`, and `FORCE_EVAL=False` prevent an accidental run or overwrite. A real training launch requires the registered arm, a passing smoke report, approved efficacy declaration, exact recipe, fresh experiment directory, supported hardware, adequate node headroom, and explicit authorisation. Do not bypass these checks.

The granularity-aware configuration trains with two concepts, `tree canopy` and `tree`, while retaining the original category-collapsed reporting annotations. Inference uses only `tree`. Primary evaluation scores individual crowns and ignores canopy groups through COCO crowd semantics. The reported checkpoint is the last-epoch final model. Boundary AP uses the instance-relative band at k=0.19; fixed 8-pixel and literature bands are comparisons.

The reported seed-42 run is complete and must not be relaunched or overwritten. To reproduce it, use a new registered run directory and follow the same preparation, smoke, efficacy, hardware, and authorisation procedure.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
if (cwd / "Core").is_dir():
    CODEBASE = cwd
elif cwd.name == "Notebook" and (cwd.parent / "Core").is_dir():
    CODEBASE = cwd.parent
else:
    candidates = [p for p in [cwd, *cwd.parents] if (p / "Codebase" / "Core").is_dir()]
    if not candidates:
        raise RuntimeError(f"Could not locate Codebase/Core from cwd={cwd}")
    CODEBASE = candidates[0] / "Codebase"

CORE = CODEBASE / "Core"
if str(CORE) not in sys.path:
    sys.path.insert(0, str(CORE))

from config import ARM_RUN_NAMES, BOUNDARY_BAND_K, FULL_FT_RUN_NAME, MASKRCNN_RUN_NAME, MODEL_BACKEND
from project_paths import A0_INFERENCE_DIR
from project_workflows import BOUNDARY_FINE_BAND_PX, MAX_DETS, create_experiment_workflow

# RUN_NAME selects the training run and its checkpoint. RUN_TAG selects the inference and
# evaluation directory. ARM is the single arm switch so the objective, run name, and
# experiment directory cannot disagree.
#   None                    : Unified-Prompt SAM3 recipe, experiments/sam3_crop_frts
#   "prompt_granularity"    : category-preserving two-prompt supervision
# Registration does not activate an arm. Launch evidence and explicit authorisation remain required.
ARM = "prompt_granularity"
PG_SMOKE_REPORT = CODEBASE / "experiments/sam3_prompt_granularity_smoke_06/smoke_report.json"
PG_EFFICACY_SPEC = CODEBASE.parent / "Context/active/prompt_granularity_efficacy.json"
PG_MICRO_BATCH = 2
PG_CHECKPOINT_BACKBONE = True
PREPARE_PG_VIEW = False
FULL_LAUNCH_AUTHORIZED = False
FULL_LAUNCH_AUTHORIZED_AT = None

if ARM is not None and MODEL_BACKEND != "maskrcnn":
    RUN_NAME = ARM_RUN_NAMES[ARM]
else:
    RUN_NAME = MASKRCNN_RUN_NAME if MODEL_BACKEND == "maskrcnn" else FULL_FT_RUN_NAME
RUN_TAG = "maskrcnn_r50" if MODEL_BACKEND == "maskrcnn" else RUN_NAME
TILE_POLICY = "tiled"
CLASS_SPLIT = "tree"
FORCE_INFERENCE = False
FORCE_EVAL = False

ANN = CODEBASE / "runtime/data/oam_tcd_instance_coco/annotations/test_annotations.coco.json"
IMAGE_ROOT = CODEBASE / "runtime/data/oam_tcd_instance_coco"

# Report final last-epoch weights. The monitoring set is also the reporting set, so selecting
# the maximum monitoring checkpoint would introduce model-selection bias.
SAM3_CHECKPOINT = CODEBASE / f"experiments/{RUN_NAME}/final_model/pytorch_model.bin"
MASKRCNN_CONFIG = CODEBASE / f"experiments/{MASKRCNN_RUN_NAME}/config.yaml"
MASKRCNN_WEIGHTS = CODEBASE / f"experiments/{MASKRCNN_RUN_NAME}/model_final.pth"

RUN_DIR = CODEBASE / f"runtime/inference_output/oam_tcd_{RUN_TAG}_{TILE_POLICY}"
PRED = RUN_DIR / "predictions/coco_predictions_segm_nms.json"
EVAL = RUN_DIR / "eval.json"
EVAL_SPLIT = RUN_DIR / f"eval_{CLASS_SPLIT}.json"
A0_DIR = A0_INFERENCE_DIR

experiment = create_experiment_workflow(RUN_NAME)

print("CODEBASE      =", CODEBASE)
print("MODEL_BACKEND =", MODEL_BACKEND)
print("RUN_NAME      =", RUN_NAME)
print("RUN_DIR       =", RUN_DIR)
print("A0_DIR        =", A0_DIR)
print("TILE_POLICY   =", TILE_POLICY, "| CLASS_SPLIT =", CLASS_SPLIT)
print("MAX_DETS      =", MAX_DETS, "| instance band k =", BOUNDARY_BAND_K)
print("Comparison bands: fixed", BOUNDARY_FINE_BAND_PX, "px and literature ratio 0.02")
print("PG smoke report:", PG_SMOKE_REPORT, "| exists:", PG_SMOKE_REPORT.is_file())
print("PG efficacy declaration:", PG_EFFICACY_SPEC, "| exists:", PG_EFFICACY_SPEC.is_file())
print("Full launch authorization:", FULL_LAUNCH_AUTHORIZED, "|", FULL_LAUNCH_AUTHORIZED_AT)

## 1. Preflight

Validate the environment and the existing category-collapsed assets. Run the strict test suite in the complete GPU environment before relying on GPU-dependent behaviour. Boundary AP is required; an unavailable dependency stops evaluation rather than producing an incomplete result. Do not regenerate the locked export during verification.

In [ ]:
experiment.validate_environment()

try:
    from boundary_iou.coco_instance_api.coco import COCO as _BoundaryCOCO  # noqa: F401
    print("boundary_iou: OK (Boundary AP will be computed)")
except ImportError as exc:
    from eval_predictions import BOUNDARY_AP_INSTALL_HINT
    raise RuntimeError(
        f"boundary_iou is not importable ({exc}). Boundary AP is the primary "
        f"metric for this scope, so fix this before running an arm:\n  "
        f"{BOUNDARY_AP_INSTALL_HINT}"
    )

In [ ]:
full_train_ann, full_test_ann = experiment.validate_assets()
full_export_dir = experiment.export_dir
summary = experiment.validate_export()

if ARM == "prompt_granularity":
    from prompt_granularity import prepare_views, validate_views
    pg_view = prepare_views(full_export_dir) if PREPARE_PG_VIEW else validate_views(full_export_dir)
    print(json.dumps(pg_view, indent=2))
    print("The training subprocess selects these separate views; ANN remains the original reporting GT.")

## 2. Data and augmentation sanity

One exported ground-truth sample, then a non-visual check that the production training pipeline carries the intended augmentation operations.

In [ ]:
from visualization import visualize_coco_ground_truth_sample

SAMPLE_SPLIT = "test"          # 'train' or 'test'
SAMPLE_IMAGE_ID = 3            # int COCO image id, or None for the first non-empty sample
MAX_GT_INSTANCES_DISPLAY = 200

ann_path = full_test_ann if SAMPLE_SPLIT == "test" else full_train_ann
fig = visualize_coco_ground_truth_sample(
    coco_json=ann_path,
    image_root=full_export_dir,
    image_id=SAMPLE_IMAGE_ID,
    max_instances=MAX_GT_INSTANCES_DISPLAY,
)
print("split:", SAMPLE_SPLIT)

In [ ]:
op_labels = experiment.augmentation_summary()

## 3. Full fine-tuning from the pretrained initialization

`LAUNCH=False` is the safe default. Every run requires a unique registered identity and a fresh experiment directory. Completed runs are protected from relaunch.

The locked recipe uses 30 epochs, seed 42, two A40 GPUs, bf16, effective image batch 16, gradient clipping 0.1, warmup fraction 0.05, cosine end ratio 0.01, and model resolution 1008. Any memory accommodation must be recorded and preserve the effective image batch.

An interrupted run is resumed explicitly with `resume_from=` and a staged checkpoint (see `project_workflows.py`). Backup, staging, and launch remain separate opt-in operations.

### Node capacity check

Run this cell before enabling a training launch on any shared, unscheduled GPU host. It displays CPU load and free GPU memory without launching a process. `launch_training` repeats the same checks and refuses a launch when the node lacks the required capacity.

In [ ]:
from device_policy import assert_node_headroom, node_load

_n = node_load()
print(f"cores            : {_n['n_cores']}")
print(f"load 1/5/15 min  : {_n['load1']:.1f} / {_n['load5']:.1f} / {_n['load15']:.1f}")
print(f"load per core    : {_n['load_per_core']:.2f}   (block threshold 0.90)")
for _g in _n["gpus"]:
    print(f"GPU {_g['index']}            : {_g['free_mib']:>6d} MiB free of {_g['total_mib']}, "
          f"{_g['util_pct']}% util")

if _n["load_per_core"] > 0.90:
    print("\n  DO NOT LAUNCH: the node has no CPU slack. A run started now would starve")
    print("  its own dataloaders and slow every other job on the node.")
elif _n["load_per_core"] > 0.60:
    print("\n  Launch will contend. Fine for a short calibration, poor for full training.")
else:
    print("\n  Clear to launch.")

if ARM == "prompt_granularity":
    _pg_headroom = assert_node_headroom(strict=True, require_gpu_mib=30830, require_gpus=2)
    print("PROMPT-GRANULARITY HEADROOM READY: True")
    print("Required free memory per GPU: 30,830 MiB")

# Print the selected objective before any costly operation. Structural guards prevent an
# arm, run name, and output directory from disagreeing; this output lets the operator confirm
# that the intended scientific objective was selected.
import config
print()
print("OBJECTIVE FOR THIS LAUNCH")
print(f"  ARM                  : {ARM}")
print(f"  run name / directory : {RUN_NAME}")
if ARM is None:
    print("  -> PLAIN BASELINE run (Unified-Prompt SAM3 recipe).")
else:
    print(f"  -> arm {ARM!r}: enabled by the launcher via "
          f"{ARM.upper()}_ENABLED_OVERRIDE; committed config stays inert.")

In [ ]:
LAUNCH = False

# SHARE_NODE may bypass the CPU-load guard only when node sharing is explicitly accepted.
# It never bypasses the GPU-memory guard and does not improve throughput. Worker reduction is
# a throughput accommodation; GPU count, effective image batch, and scientific recipe remain fixed.
SHARE_NODE = False
REDUCED_WORKERS = 4 if SHARE_NODE else None
REDUCED_MICRO_BATCH = 2 if SHARE_NODE else None

# Prompt granularity uses microbatch 2, accumulation 4, and two ranks for effective image
# batch 16. The launcher takes the peak reserved memory per rank from the smoke report, adds a
# 4,096 MiB margin, and requires that much free memory on each A40.
pg_options = {}
PG_LAUNCH_READY = ARM != "prompt_granularity"
if ARM == "prompt_granularity":
    pg_options = dict(
        smoke_report=Path(PG_SMOKE_REPORT) if PG_SMOKE_REPORT else None,
        efficacy_spec=Path(PG_EFFICACY_SPEC) if PG_EFFICACY_SPEC else None,
    )
    REDUCED_MICRO_BATCH = PG_MICRO_BATCH
    REDUCED_WORKERS = 4
    PG_LAUNCH_READY = bool(
        FULL_LAUNCH_AUTHORIZED
        and PG_SMOKE_REPORT
        and Path(PG_SMOKE_REPORT).is_file()
        and PG_EFFICACY_SPEC
        and Path(PG_EFFICACY_SPEC).is_file()
        and PG_MICRO_BATCH == 2
        and PG_CHECKPOINT_BACKBONE is True
        and RUN_NAME == "sam3_prompt_granularity"
    )
    print("PROMPT-GRANULARITY NOTEBOOK LAUNCH READY:", PG_LAUNCH_READY)

if LAUNCH and not PG_LAUNCH_READY:
    raise RuntimeError("Notebook launch is not ready; do not bypass smoke, efficacy, recipe, identity, or authorization checks.")

training_log_md = experiment.launch_training(
    LAUNCH,
    arm=ARM,
    require_node_headroom=not SHARE_NODE,
    train_workers=REDUCED_WORKERS,
    val_workers=2 if SHARE_NODE or ARM == "prompt_granularity" else None,
    micro_batch=REDUCED_MICRO_BATCH,
    # Whole-backbone activation checkpointing is numerically exact with preserved RNG state.
    # It exchanges additional computation for lower activation memory.
    checkpoint_backbone=PG_CHECKPOINT_BACKBONE if ARM == "prompt_granularity" else SHARE_NODE,
    **pg_options,
)
full_log_dir = experiment.experiment_dir

### Training-complete gate

Every later cell requires a trained final model. This gate distinguishes a completed run, an interrupted run with a recoverable checkpoint, and an inert notebook execution with `LAUNCH=False`. It stops before inference when final weights are unavailable.

In [ ]:
CKPT = SAM3_CHECKPOINT if MODEL_BACKEND == "sam3" else MASKRCNN_WEIGHTS
MID_RUN = experiment.experiment_dir / "checkpoints" / "checkpoint.pt"

print(f"run       : {RUN_NAME}")
print(f"directory : {experiment.experiment_dir}")
print(f"LAUNCH    : {LAUNCH}\n")

if CKPT.is_file():
    _mb = CKPT.stat().st_size / 1e6
    print(f"  Final weights present: {CKPT} ({_mb:.0f} MB).")
    print("  Verify the run completion and provenance record before inference and evaluation.")
elif MODEL_BACKEND == "sam3" and MID_RUN.is_file():
    from project_workflows import read_checkpoint_epoch

    _ep = read_checkpoint_epoch(MID_RUN)
    raise RuntimeError(
        f"Training is incomplete: a checkpoint exists at epoch {_ep}, but final_model/ "
        f"has not been exported. Do not substitute a mid-run checkpoint for the reported "
        f"model. Resume the run explicitly with resume_from= (see project_workflows.py), RUN_NAME={RUN_NAME!r}."
    )
elif MODEL_BACKEND == "maskrcnn" and (experiment.experiment_dir / "last_checkpoint").is_file():
    raise RuntimeError("Mask R-CNN has a checkpoint marker but no model_final.pth; inspect its training record before recovery.")
elif not LAUNCH:
    raise RuntimeError(
        f"Training has not run because LAUNCH is False, and {experiment.experiment_dir} "
        f"contains no final model. Complete the required gates before enabling a new launch."
    )
else:
    raise RuntimeError(
        f"LAUNCH is True but no final model was exported in {experiment.experiment_dir}. "
        f"Inspect the launch output before attempting inference."
    )

In [ ]:
log_report = experiment.inspect_logs()

## 4. Lock the TEST protocol

Requires the 439-image export, 30,646 annotations, native 2048x2048, the collapsed `tree` category, and source-split metadata. Records annotation and image-ID hashes next to the result so a row can always be traced to the export that produced it.

In [ ]:
from oam_tcd_protocol import validate_oam_tcd_test

protocol = validate_oam_tcd_test(ANN)
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / "oam_tcd_protocol.json").write_text(json.dumps(protocol, indent=2) + "\n")
protocol

## 5. Inference

SAM3: tiled 1024/256 or whole-image at 1008px, no test-time augmentation. Mask R-CNN R50: whole-image matches the paper; tiled is the like-for-like comparison.

Skipped when a dump already exists unless `FORCE_INFERENCE = True`. Boundary AP is a function of the dump and the GT only, so **refreshing a Boundary AP record on an existing dump does not require re-running inference.**

In [ ]:
from project_workflows import run_tiled_inference

# One definition of the locked protocol
# (tile 1024 / overlap 256 / dedup 0.30 / score-thresh 0.0 / maxdets 512), so a
# fresh arm and a resumed arm cannot produce dumps that differ by a stray flag.
run_tiled_inference(
    out_path=PRED,
    ann=ANN,
    image_root=IMAGE_ROOT,
    checkpoint=SAM3_CHECKPOINT if MODEL_BACKEND == "sam3" else None,
    tile_policy=TILE_POLICY,
    backend=MODEL_BACKEND,
    detectron_config=MASKRCNN_CONFIG,
    detectron_weights=MASKRCNN_WEIGHTS,
    force=FORCE_INFERENCE,
)

## 6. Evaluate

Tree-only (`--class-split tree`) is the primary reporting protocol; canopy groups retain COCO crowd-ignore semantics. The collapsed row (`--class-split none`) is a separate comparison and diagnostic view, never a replacement for tree-only results.

All three Boundary AP variants are retained: **instance-relative k=0.19** is primary, fixed 8px is a comparison, and image-diagonal ratio 0.02 is the literature comparison. The shared helper passes k explicitly and rejects a result missing that metric. Existing current evaluations are read without modification; stale or incompatible records require an explicit decision before recomputation. `BAND_SWEEP=False` and `FORCE_EVAL=False` remain the defaults.

In [ ]:
from project_workflows import run_evaluation

BAND_SWEEP = False

eval_all = run_evaluation(pred=PRED, gt=ANN, out_path=EVAL, class_split="none", band_sweep=BAND_SWEEP, force=FORCE_EVAL)
eval_split = (
    run_evaluation(pred=PRED, gt=ANN, out_path=EVAL_SPLIT, class_split=CLASS_SPLIT, force=FORCE_EVAL)
    if CLASS_SPLIT != "none" else eval_all
)

eval_split.get("coco_eval")

## 7. Visualize predictions

One best-B-IoU image per GT-density bucket (~50, ~100, ~300, ~500+ crowns), rendered as Image | GT | Prediction and saved under the run's `visualizations/`. Keeping this in the same notebook means every arm leaves behind a visual record next to its numbers.

In [ ]:
from visualization import pick_dense_biou_image_ids, visualize_dumped_prediction
from config import VIZ_SCORE_THRESHOLD
import matplotlib.pyplot as plt

VIS_DIR = RUN_DIR / "visualizations"
VIS_DIR.mkdir(parents=True, exist_ok=True)

buckets = [(50, 100), (100, 300), (300, 500), (500, None)]
bucket_ids = pick_dense_biou_image_ids(
    coco_json=str(ANN),
    pred_json=str(PRED),
    buckets=buckets,
    per_bucket=1,
    score_threshold=VIZ_SCORE_THRESHOLD,
)

all_ids = []
for label, ids in bucket_ids.items():
    print(f"Bucket {label}: {ids}")
    all_ids.extend(ids)

for image_id in all_ids:
    fig = visualize_dumped_prediction(
        coco_json=str(ANN),
        image_root=str(IMAGE_ROOT),
        image_id=image_id,
        pred_json=str(PRED),
        label=f"{RUN_TAG} {TILE_POLICY}",
        score_threshold=VIZ_SCORE_THRESHOLD,
    )
    out_path = VIS_DIR / f"vis_{image_id}.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved -> {out_path}")
    plt.show()

## 8. Compare with the locked Unified-Prompt SAM3 baseline

Verify inference comparability before reading a model delta. The baseline is always `oam_tcd_sam3_crop_frts_tiled`; no alternative directory is used as a fallback. Reference files are read-only.

The table reports Mask AP, size bins, instance-relative Boundary AP at k=0.19, and the fixed 8-pixel and literature comparison bands. Apply the approved efficacy declaration to the exact tree-split values.

In [ ]:
import pandas as pd

from device_policy import check_dumps_comparable
from project_workflows import read_metric_row

if MODEL_BACKEND == "sam3":
    check_dumps_comparable(A0_DIR, RUN_DIR)

SUFFIX = "eval.json" if CLASS_SPLIT == "none" else f"eval_{CLASS_SPLIT}.json"

REFERENCE_RUNS = {
    "Unified-Prompt SAM3 tiled": A0_DIR / SUFFIX,
    "Mask R-CNN R50 tiled": CODEBASE / f"runtime/inference_output/oam_tcd_maskrcnn_r50_tiled/{SUFFIX}",
    "Mask R-CNN R50 whole-image": CODEBASE / f"runtime/inference_output/oam_tcd_maskrcnn_r50_whole-image/{SUFFIX}",
}

this_label = f"THIS RUN: {RUN_TAG} {TILE_POLICY}"
rows = [read_metric_row(this_label, EVAL_SPLIT if CLASS_SPLIT != "none" else EVAL, class_split=CLASS_SPLIT)]
rows += [read_metric_row(label, path, class_split=CLASS_SPLIT) for label, path in REFERENCE_RUNS.items()]
table = pd.DataFrame([row for row in rows if row is not None]).set_index("run")
display(table.round(4))

a0_label = "Unified-Prompt SAM3 tiled"
if RUN_DIR.resolve() != A0_DIR.resolve() and this_label in table.index and a0_label in table.index:
    delta = table.loc[this_label] - table.loc[a0_label]
    print(f"\nDelta vs Unified-Prompt SAM3 (class split: {CLASS_SPLIT}, instance band k={BOUNDARY_BAND_K})")
    print(delta.round(4).to_string())
    print(f"\nMask AP delta: {delta['Mask AP']:+.4f}; Boundary AP instance delta: {delta['Bnd AP instance']:+.4f}")
    print("Apply the approved efficacy declaration to these exact values.")
else:
    print("\nThis is the baseline itself or a required comparison row is missing; no arm verdict is available.")